# Excel workbook comparison — values only, sheet by sheet, cell by cell

Compares **two Excel workbooks** and writes a third workbook containing every difference.

**What it compares**
- Every worksheet, matched by sheet name (sheets only in one workbook are reported, not diffed).
- Every populated cell in each matched sheet, compared by **grid position** (`A1` vs `A1`).
- **Cached values only** — a cell holding `=SUM(A1:A9)` is compared on the number Excel last calculated,
  never on the formula text. Two cells with different formulas that produce `42` are treated as equal.

**What it ignores**
- Charts, graphs, shapes, images, chart sheets, pivot caches.
- Formatting: fonts, fills, number formats, column widths, conditional formatting, borders.
- Comments, defined names, macros, data validation, protection.

**Formats** — `.xlsx`, `.xlsm`, `.xlsb` and `.xls`, and the two sides don't have to match.
`.xlsb` and `.xls` are converted to `.xlsx` first when Excel or LibreOffice is available, and otherwise
read directly with `pyxlsb` / `xlrd`. See *Reading .xlsb* below — the direct path has one real caveat.

**Output** — one `.xlsx` with:
| Sheet | Contents |
|---|---|
| `Summary` | Run metadata, which reader handled each file, options used, totals, pass/fail verdict |
| `Sheet Inventory` | Every sheet in either workbook: present where, used range, diff count |
| `Cell Differences` | One row per differing cell: sheet, cell ref, both values, both types, delta |
| `<sheet name>` tabs | Optional per-sheet breakdown (`per_sheet_tabs=True`) |

---

### Read this before trusting the output

1. **Cached values must exist.** `openpyxl` cannot calculate formulas; it reads the value Excel stored on
   the last save. A workbook produced by a script (pandas, openpyxl, some export tools) often has **no**
   cached values, so every formula cell reads as blank. The notebook detects this and warns you — the fix
   is to open the workbook in Excel/LibreOffice and re-save it once.
2. **Comparison is positional.** Inserting a row near the top of a sheet shifts everything below it, so a
   one-row insert can report thousands of differences. That is correct behaviour for a cell-over-cell
   diff, not a bug. For row-keyed reconciliation you want a different tool (a join on a key column).

### Reading `.xlsb`

The binary format stores no cell types, so **`pyxlsb` hands dates back as raw Excel serial numbers**
(`45842.0`, not `2025-07-04`) and cannot see formulas at all. Consequences:

- **Two `.xlsb` files** compare fine — both sides speak serials, and they line up.
- **`.xlsb` vs `.xlsx`** would compare `45870.0` against a real date and call every date different, so
  `date_handling="auto"` converts *both* sides to serial numbers. Dates then compare correctly; the
  report just shows that side's dates as numbers.
- **The uncalculated-formula check is skipped** for a `.xlsb` read this way.

All three go away if the file is converted to `.xlsx` first, which the notebook attempts automatically.
To do it by hand: open in Excel, *File → Save As → Excel Workbook (.xlsx)*.

Requirements: `pip install openpyxl pandas` plus `pyxlsb` for `.xlsb` and `xlrd` for `.xls`.

## 1. Imports and the options schema
Run once. The printout lists which formats this environment can read — `.xlsx`/`.xlsm` always,
`.xlsb` if `pyxlsb` is installed, `.xls` if `xlrd` is. `Options` documents every comparison rule.

In [ ]:
from __future__ import annotations

import datetime as dt
import math
import os
import re
import shutil
import subprocess
import sys
import tempfile
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import openpyxl
from openpyxl import Workbook, load_workbook
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

print(f"python     {sys.version.split()[0]}")
print(f"openpyxl   {openpyxl.__version__}   .xlsx .xlsm")

try:
    import pyxlsb                                  # binary .xlsb
except ImportError:
    pyxlsb = None
    print("pyxlsb     not installed             .xlsb -> pip install pyxlsb")
else:
    print(f"pyxlsb     {pyxlsb.__version__}{' ' * max(1, 10 - len(pyxlsb.__version__))}.xlsb")

try:
    import xlrd                                    # legacy binary .xls
except ImportError:
    xlrd = None
    print("xlrd       not installed             .xls  -> pip install xlrd")
else:
    print(f"xlrd       {xlrd.__version__}{' ' * max(1, 10 - len(xlrd.__version__))}.xls")

try:
    import pandas as pd                            # only used by the exploration section
except ImportError:
    pd = None
    print("pandas     not installed             (section 6 will be skipped)")
else:
    print(f"pandas     {pd.__version__}")


@dataclass
class Options:
    """Every knob for the comparison. See section 2 for the instance you actually edit."""

    # --- file formats --------------------------------------------------------
    convert_binary_formats: str = "auto"          # "auto" | "excel" | "libreoffice" | "never"
    date_handling: str = "auto"                   # "auto" | "native" | "serial"
    dates_1904: bool = False                      # workbooks saved with the 1904 epoch (rare)

    # --- which sheets --------------------------------------------------------
    only_sheets: Optional[Sequence[str]] = None   # None = every sheet; else an allow-list
    ignore_sheets: Sequence[str] = ()             # sheet names to skip entirely
    sheet_map: Dict[str, str] = field(default_factory=dict)  # {name_in_A: name_in_B} for renamed sheets
    match_sheets_case_insensitively: bool = True  # "Data" in A pairs with "DATA" in B
    include_hidden_sheets: bool = True

    # --- which cells ---------------------------------------------------------
    max_rows: Optional[int] = None                # hard cap on rows scanned per sheet (None = all)
    max_cols: Optional[int] = None                # hard cap on columns scanned per sheet

    # --- what counts as "the same value" -------------------------------------
    trim_whitespace: bool = True                  # "  abc " == "abc"
    collapse_whitespace: bool = False             # "a    b" == "a b"
    case_insensitive_text: bool = False           # "Total" == "total"
    blank_equals_empty_string: bool = True        # empty cell == cell containing ""
    numeric_strings_as_numbers: bool = False      # "1,234.50" (text) == 1234.5 (number)
    abs_tolerance: float = 1e-9                   # absolute float tolerance
    rel_tolerance: float = 0.0                    # relative float tolerance, e.g. 1e-9
    round_numbers_to: Optional[int] = None        # round both sides to N decimals before comparing
    ignore_time_component: bool = False           # 2024-01-01 00:00 == 2024-01-01 09:30

    # --- safety valves -------------------------------------------------------
    max_diffs_per_sheet: int = 100_000
    max_total_diffs: int = 500_000
    check_cached_values: bool = True              # second pass to warn about uncalculated formulas

    # --- output --------------------------------------------------------------
    per_sheet_tabs: bool = False                  # one extra tab per sheet that has differences
    max_value_chars: int = 500                    # truncate long cell values in the report
    freeze_and_filter: bool = True                # freeze header row + add autofilter


EXCEL_ERRORS = {
    "#N/A", "#VALUE!", "#REF!", "#DIV/0!", "#NUM!", "#NAME?",
    "#NULL!", "#SPILL!", "#CALC!", "#GETTING_DATA", "#FIELD!", "#UNKNOWN!",
}

# BIFF12 error codes, as pyxlsb hands them back (hex strings)
XLSB_ERRORS = {
    "0x0": "#NULL!", "0x7": "#DIV/0!", "0xf": "#VALUE!", "0x17": "#REF!",
    "0x1d": "#NAME?", "0x24": "#NUM!", "0x2a": "#N/A", "0x2b": "#GETTING_DATA",
}

## 2. Settings
The only cell you need to touch. Paths can be relative to this notebook or absolute, and may be
`.xlsx`, `.xlsm`, `.xlsb` or `.xls` — mixed formats on the two sides are fine.

Four rules worth calling out:
- **`convert_binary_formats`** — for `.xlsb`/`.xls`, try Excel then LibreOffice to convert to `.xlsx`
  first. Worth leaving on `"auto"`: a converted file carries real dates and real cached values, which
  a direct binary read cannot. Set `"never"` to force the direct read.
- **`date_handling`** — `"auto"` compares dates as Excel serial numbers whenever one side is read by
  `pyxlsb` (which reports dates as bare numbers) and the other isn't, so dates still match across a
  mixed `.xlsb` vs `.xlsx` comparison. `"native"` compares them as read; `"serial"` always converts.
- **`abs_tolerance`** — floats that came from different calculation paths often differ in the 15th
  decimal. The default `1e-9` absorbs that. Set it to `0.005` to ignore anything under half a cent,
  or `0.0` to demand bit-exact equality.
- **`numeric_strings_as_numbers`** — off by default, so the text `"1,234.50"` is reported as different
  from the number `1234.5`. That distinction usually matters; turn it on when it doesn't.

In [ ]:
# ============================================================================
#  EDIT THIS CELL
# ============================================================================

FILE_A = "workbook_a.xlsx"          # the "before" / baseline workbook
FILE_B = "workbook_b.xlsx"          # the "after" / candidate workbook
OUTPUT_FILE = "differences.xlsx"    # report written here (overwritten if it exists)

OPT = Options(
    # File formats ---------------------------------------------------------
    convert_binary_formats="auto",       # "auto" (Excel, then LibreOffice) | "excel"
                                         # | "libreoffice" | "never" (read .xlsb/.xls directly)
    date_handling="auto",                # "auto" | "native" | "serial"
    dates_1904=False,                    # only for workbooks saved with the 1904 epoch

    # Sheets ---------------------------------------------------------------
    only_sheets=None,                    # e.g. ["Summary", "Inputs"]
    ignore_sheets=(),                    # e.g. ["Notes", "Scratch"]
    sheet_map={},                        # e.g. {"Q1 Data": "Q1 Data (rev)"}
    match_sheets_case_insensitively=True,
    include_hidden_sheets=True,

    # Cells ----------------------------------------------------------------
    max_rows=None,
    max_cols=None,

    # Equality rules -------------------------------------------------------
    trim_whitespace=True,
    collapse_whitespace=False,
    case_insensitive_text=False,
    blank_equals_empty_string=True,
    numeric_strings_as_numbers=False,
    abs_tolerance=1e-9,                  # raise to e.g. 0.005 to ignore sub-cent drift
    rel_tolerance=0.0,                   # or use 1e-9 for scale-independent comparison
    round_numbers_to=None,               # e.g. 2 to compare everything to 2dp
    ignore_time_component=False,

    # Safety valves --------------------------------------------------------
    max_diffs_per_sheet=100_000,
    max_total_diffs=500_000,
    check_cached_values=True,

    # Report ---------------------------------------------------------------
    per_sheet_tabs=False,
    max_value_chars=500,
    freeze_and_filter=True,
)

## 3. Readers — one per file format
`.xlsx`/`.xlsm` go straight to openpyxl. `.xlsb` and `.xls` are first offered to Excel (via COM) and
then LibreOffice for conversion to `.xlsx`, because a converted file keeps real dates and real cached
values; if neither converter is available they fall back to reading the binary directly with `pyxlsb`
or `xlrd`. Whichever path ran is recorded in the report. No edits needed here.

In [ ]:
# ---------------------------------------------------------------------------
#  Converting binary formats to .xlsx (best fidelity when Excel or LibreOffice is around)
# ---------------------------------------------------------------------------

_CONVERT_DIR: Optional[tempfile.TemporaryDirectory] = None
_CONVERT_CACHE: Dict[Tuple[str, float], Path] = {}
_CONVERT_FAILED: Dict[Tuple[str, float], str] = {}      # so a slow converter is only tried once


def _convert_dir() -> Path:
    global _CONVERT_DIR
    if _CONVERT_DIR is None:
        _CONVERT_DIR = tempfile.TemporaryDirectory(prefix="xlsx_convert_")
    return Path(_CONVERT_DIR.name)


def convert_with_excel(src: Path, dest: Path) -> Path:
    """Windows + Excel installed. Highest fidelity: real dates, real cached values."""
    import win32com.client as win32                          # pywin32

    excel = win32.DispatchEx("Excel.Application")
    excel.Visible = False
    excel.DisplayAlerts = False
    excel.AskToUpdateLinks = False
    try:
        wb = excel.Workbooks.Open(str(src.resolve()), UpdateLinks=0, ReadOnly=True)
        try:
            wb.SaveAs(str(dest.resolve()), FileFormat=51)     # 51 = xlOpenXMLWorkbook
        finally:
            wb.Close(SaveChanges=False)
    finally:
        excel.Quit()
    return dest


def convert_with_libreoffice(src: Path, dest: Path) -> Path:
    soffice = shutil.which("soffice") or shutil.which("libreoffice")
    if soffice is None:
        raise RuntimeError("soffice/libreoffice not on PATH")
    profile = dest.parent / "lo_profile"
    subprocess.run(
        [soffice, "--headless", "--norestore",
         f"-env:UserInstallation=file://{profile}",
         "--convert-to", "xlsx", "--outdir", str(dest.parent), str(src.resolve())],
        check=True, capture_output=True, timeout=600,
    )
    produced = dest.parent / (src.stem + ".xlsx")
    if not produced.exists():
        raise RuntimeError("LibreOffice reported success but produced no .xlsx")
    if produced != dest:
        produced.replace(dest)
    return dest


def convert_to_xlsx(src: Path, opt: Options) -> Tuple[Optional[Path], str]:
    """Try to convert a binary workbook to .xlsx. Returns (path or None, explanation)."""
    if opt.convert_binary_formats == "never":
        return None, "conversion disabled (convert_binary_formats='never')"

    key = (str(src.resolve()), src.stat().st_mtime)
    if key in _CONVERT_CACHE and _CONVERT_CACHE[key].exists():
        return _CONVERT_CACHE[key], "reused an earlier conversion from this session"
    if key in _CONVERT_FAILED:
        return None, _CONVERT_FAILED[key]

    dest = _convert_dir() / (re.sub(r"[^\w.-]+", "_", src.stem) + ".xlsx")
    wanted = opt.convert_binary_formats
    attempts = []
    if wanted in ("auto", "excel"):
        attempts.append(("Excel", convert_with_excel))
    if wanted in ("auto", "libreoffice"):
        attempts.append(("LibreOffice", convert_with_libreoffice))

    notes = []
    for label, fn in attempts:
        try:
            out = fn(src, dest)
        except Exception as exc:                              # not installed, timed out, refused...
            notes.append(f"{label}: {type(exc).__name__}: {str(exc).splitlines()[0][:120]}")
            continue
        _CONVERT_CACHE[key] = out
        return out, f"converted to .xlsx with {label}"
    reason = "; ".join(notes) or "no converter configured"
    _CONVERT_FAILED[key] = reason
    return None, reason


# ---------------------------------------------------------------------------
#  One reader per format. They all expose the same three things:
#    .sheets / .chart_sheets / .hidden_sheets, .grid(name), .formula_scan(name, grid)
# ---------------------------------------------------------------------------

class WorkbookSource:
    engine = "?"
    typed_dates = True        # False when the format hands dates back as raw serial numbers
    has_formulas = True       # False when the reader cannot see formula text at all

    def __init__(self, path: Path, opt: Options):
        self.path = Path(path)
        self.opt = opt
        self.origin: Optional[Path] = None      # set when this came from a conversion
        self.note = ""
        self.sheets: List[str] = []
        self.chart_sheets: List[str] = []
        self.hidden_sheets: List[str] = []

    def grid(self, name: str) -> Tuple[Dict[Tuple[int, int], Any], int, int]:
        raise NotImplementedError

    def formula_scan(self, name: str, values: Dict[Tuple[int, int], Any]):
        return 0, 0, []

    def close(self):
        pass

    @property
    def label(self) -> str:
        src = f"{self.origin.name} -> " if self.origin else ""
        return f"{src}{self.path.name} [{self.engine}]"


class OpenpyxlSource(WorkbookSource):
    engine = "openpyxl"

    def __init__(self, path, opt):
        super().__init__(path, opt)
        self.wb = load_workbook(self.path, data_only=True, read_only=True, keep_links=False)
        self.wbf = (load_workbook(self.path, data_only=False, read_only=True, keep_links=False)
                    if opt.check_cached_values else None)
        for name in self.wb.sheetnames:
            ws = self.wb[name]
            if not hasattr(ws, "iter_rows"):                  # chart sheet / dialog sheet
                self.chart_sheets.append(name)
            elif getattr(ws, "sheet_state", "visible") != "visible" and not opt.include_hidden_sheets:
                self.hidden_sheets.append(name)
            else:
                self.sheets.append(name)

    def grid(self, name):
        g, mr, mc = {}, 0, 0
        for row in self.wb[name].iter_rows(max_row=self.opt.max_rows, max_col=self.opt.max_cols):
            for cell in row:
                v = cell.value
                if v is None or (isinstance(v, str) and v.strip() == "" and self.opt.blank_equals_empty_string):
                    continue
                g[(cell.row, cell.column)] = v
                mr, mc = max(mr, cell.row), max(mc, cell.column)
        return g, mr, mc

    def formula_scan(self, name, values):
        if self.wbf is None:
            return 0, 0, []
        total = missing = 0
        samples: List[str] = []
        for row in self.wbf[name].iter_rows(max_row=self.opt.max_rows, max_col=self.opt.max_cols):
            for cell in row:
                v = cell.value
                if not (isinstance(v, str) and v.startswith("=")
                        or type(v).__name__ in {"ArrayFormula", "DataTableFormula"}):
                    continue
                total += 1
                if (cell.row, cell.column) not in values:
                    missing += 1
                    if len(samples) < 5:
                        samples.append(f"{get_column_letter(cell.column)}{cell.row}")
        return total, missing, samples

    def close(self):
        self.wb.close()
        if self.wbf is not None:
            self.wbf.close()


class PyxlsbSource(WorkbookSource):
    """.xlsb via pyxlsb. Values only - the format gives us no cell types and no formulas."""

    engine = "pyxlsb"
    typed_dates = False
    has_formulas = False

    def __init__(self, path, opt):
        super().__init__(path, opt)
        if pyxlsb is None:
            raise ImportError("Reading .xlsb needs pyxlsb: pip install pyxlsb")
        self.wb = pyxlsb.open_workbook(str(self.path))
        chart_targets = {}
        try:                                                   # private, so treat as best-effort
            chart_targets = {n: t for n, t in getattr(self.wb, "_sheets", [])}
        except Exception:
            pass
        for name in self.wb.sheets:
            if "chartsheet" in str(chart_targets.get(name, "")).lower():
                self.chart_sheets.append(name)
            else:
                self.sheets.append(name)
        self.note = "sheet visibility is not exposed by pyxlsb, so hidden sheets are included"

    def grid(self, name):
        g, mr, mc = {}, 0, 0
        max_r, max_c = self.opt.max_rows, self.opt.max_cols
        with self.wb.get_sheet(name) as sh:
            for row in sh.rows(sparse=True):
                for cell in row:
                    v = cell.v
                    if v is None or (isinstance(v, str) and v.strip() == ""
                                     and self.opt.blank_equals_empty_string):
                        continue
                    r, c = cell.r + 1, cell.c + 1              # pyxlsb is 0-indexed
                    if (max_r and r > max_r) or (max_c and c > max_c):
                        continue
                    if isinstance(v, str) and v in XLSB_ERRORS:
                        v = XLSB_ERRORS[v]                     # '0x2a' -> '#N/A'
                    g[(r, c)] = v
                    mr, mc = max(mr, r), max(mc, c)
        return g, mr, mc

    def close(self):
        self.wb.close()


class XlrdSource(WorkbookSource):
    """Legacy .xls via xlrd. Cached values, with proper date and error typing."""

    engine = "xlrd"
    has_formulas = False

    def __init__(self, path, opt):
        super().__init__(path, opt)
        if xlrd is None:
            raise ImportError("Reading .xls needs xlrd: pip install xlrd")
        self.book = xlrd.open_workbook(str(self.path), on_demand=True)
        for name in self.book.sheet_names():
            sh = self.book.sheet_by_name(name)
            if getattr(sh, "visibility", 0) != 0 and not opt.include_hidden_sheets:
                self.hidden_sheets.append(name)
            else:
                self.sheets.append(name)
            self.book.unload_sheet(name)

    def grid(self, name):
        sh = self.book.sheet_by_name(name)
        g, mr, mc = {}, 0, 0
        n_rows = min(sh.nrows, self.opt.max_rows or sh.nrows)
        n_cols = min(sh.ncols, self.opt.max_cols or sh.ncols)
        for r in range(n_rows):
            for c in range(n_cols):
                ct, v = sh.cell_type(r, c), sh.cell_value(r, c)
                if ct in (xlrd.XL_CELL_EMPTY, xlrd.XL_CELL_BLANK):
                    continue
                if ct == xlrd.XL_CELL_DATE:
                    try:
                        v = xlrd.xldate.xldate_as_datetime(v, self.book.datemode)
                    except Exception:
                        pass
                elif ct == xlrd.XL_CELL_BOOLEAN:
                    v = bool(v)
                elif ct == xlrd.XL_CELL_ERROR:
                    v = xlrd.error_text_from_code.get(v, f"#ERR:{v}")
                elif isinstance(v, str) and v.strip() == "" and self.opt.blank_equals_empty_string:
                    continue
                g[(r + 1, c + 1)] = v
                mr, mc = max(mr, r + 1), max(mc, c + 1)
        self.book.unload_sheet(name)
        return g, mr, mc

    def close(self):
        self.book.release_resources()


READABLE = {".xlsx", ".xlsm", ".xltx", ".xltm", ".xlsb", ".xls"}


def open_source(path: str | os.PathLike, opt: Options) -> WorkbookSource:
    """Pick a reader for this file, converting binary formats to .xlsx first when we can."""
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Workbook not found: {p.resolve()}")
    suffix = p.suffix.lower()
    if suffix not in READABLE:
        raise ValueError(f"{p.name}: unsupported extension '{suffix}'. Expected one of {sorted(READABLE)}.")

    if suffix in {".xlsb", ".xls"}:
        converted, why = convert_to_xlsx(p, opt)
        if converted is not None:
            src = OpenpyxlSource(converted, opt)
            src.origin, src.note = p, why
            return src
        fallback = PyxlsbSource if suffix == ".xlsb" else XlrdSource
        src = fallback(p, opt)
        src.note = f"{why}; read directly with {src.engine}" + (f". {src.note}" if src.note else "")
        return src

    return OpenpyxlSource(p, opt)

## 4. Comparison engine
Pairs the sheets by name, then walks the union of populated cells in each pair. No edits needed.

In [ ]:
# ---------------------------------------------------------------------------
#  Value semantics
# ---------------------------------------------------------------------------

_NUM_RE = re.compile(r"^[\s$€£]*-?[\d,]*\.?\d+\s*%?$")


def value_kind(v: Any) -> str:
    """The type as actually read - reported verbatim in the 'Type in A/B' columns."""
    if v is None:
        return "blank"
    if isinstance(v, bool):
        return "boolean"
    if isinstance(v, (int, float)):
        return "number"
    if isinstance(v, dt.datetime):
        return "datetime"
    if isinstance(v, dt.date):
        return "date"
    if isinstance(v, dt.time):
        return "time"
    if isinstance(v, dt.timedelta):
        return "duration"
    if isinstance(v, str):
        return "error" if v.strip().upper() in EXCEL_ERRORS else "text"
    return type(v).__name__


def to_excel_serial(v: Any, dates_1904: bool = False) -> float:
    """Date/datetime -> the number Excel stores for it. Accurate for 1900-03-01 onwards."""
    base = dt.datetime(1904, 1, 1) if dates_1904 else dt.datetime(1899, 12, 30)
    if isinstance(v, dt.datetime):
        moment = v.replace(tzinfo=None)
    else:
        moment = dt.datetime.combine(v, dt.time.min)
    delta = moment - base
    return delta.days + delta.seconds / 86400 + delta.microseconds / 86_400_000_000


def parse_numeric_string(s: str) -> Optional[float]:
    """'1,234.50' -> 1234.5, '12%' -> 0.12, '$9' -> 9.0. None when it isn't a number."""
    if not _NUM_RE.match(s):
        return None
    cleaned = s.strip().lstrip("$€£").replace(",", "").strip()
    pct = cleaned.endswith("%")
    if pct:
        cleaned = cleaned[:-1].strip()
    try:
        n = float(cleaned)
    except ValueError:
        return None
    return n / 100.0 if pct else n


def normalize(v: Any, opt: Options) -> Any:
    """Apply the equality rules, producing a canonical comparable value."""
    if v is None:
        return None

    if isinstance(v, str):
        s = v
        if opt.trim_whitespace:
            s = s.strip()
        if opt.collapse_whitespace:
            s = re.sub(r"\s+", " ", s)
        if s == "" and opt.blank_equals_empty_string:
            return None
        if opt.numeric_strings_as_numbers:
            n = parse_numeric_string(s)
            if n is not None:
                return round(n, opt.round_numbers_to) if opt.round_numbers_to is not None else n
        return s.casefold() if opt.case_insensitive_text else s

    if isinstance(v, bool):
        return v

    if isinstance(v, (dt.date, dt.datetime)):
        if opt.date_handling == "serial":
            n = to_excel_serial(v, opt.dates_1904)
            return math.floor(n) if opt.ignore_time_component else n
        if isinstance(v, dt.datetime) and opt.ignore_time_component:
            return v.date()
        return v

    if isinstance(v, (int, float)):
        f = float(v)
        if math.isnan(f):
            return None
        if opt.date_handling == "serial" and opt.ignore_time_component:
            f = math.floor(f)                      # a bare serial compared day-for-day
        if opt.round_numbers_to is not None:
            f = round(f, opt.round_numbers_to)
        return f

    return v


def values_equal(a: Any, b: Any, opt: Options) -> bool:
    na, nb = normalize(a, opt), normalize(b, opt)

    if na is None or nb is None:
        return na is None and nb is None

    a_bool, b_bool = isinstance(na, bool), isinstance(nb, bool)
    if a_bool or b_bool:
        return a_bool and b_bool and na == nb

    if isinstance(na, (int, float)) and isinstance(nb, (int, float)):
        return math.isclose(na, nb, rel_tol=opt.rel_tolerance, abs_tol=opt.abs_tolerance)

    if isinstance(na, (dt.date, dt.datetime)) and isinstance(nb, (dt.date, dt.datetime)):
        at = na if isinstance(na, dt.datetime) else dt.datetime.combine(na, dt.time.min)
        bt = nb if isinstance(nb, dt.datetime) else dt.datetime.combine(nb, dt.time.min)
        return at.date() == bt.date() if opt.ignore_time_component else at == bt

    if type(na) is not type(nb):
        return False
    return na == nb


def difference_kind(va: Any, vb: Any, opt: Options) -> str:
    """Label the difference, using the domain the values were actually compared in."""
    ka, kb = value_kind(va), value_kind(vb)
    if ka == "blank":
        return "Added in B"
    if kb == "blank":
        return "Removed in B"
    if opt.date_handling == "serial":               # dates were compared as numbers on both sides
        ka = "number" if ka in {"date", "datetime"} else ka
        kb = "number" if kb in {"date", "datetime"} else kb
    return "Value changed" if ka == kb else "Type changed"


# ---------------------------------------------------------------------------
#  Sheet pairing
# ---------------------------------------------------------------------------

def pair_sheets(names_a: Sequence[str], names_b: Sequence[str], opt: Options):
    """Return (pairs, only_in_a, only_in_b) where pairs is [(name_a, name_b), ...]."""
    def keep(n: str) -> bool:
        if opt.only_sheets is not None and n not in opt.only_sheets:
            return False
        return n not in set(opt.ignore_sheets)

    a_list = [n for n in names_a if keep(n)]
    b_pool = list(names_b)

    def lookup(target: str) -> Optional[str]:
        if target in b_pool:
            return target
        if opt.match_sheets_case_insensitively:
            for n in b_pool:
                if n.casefold() == target.casefold():
                    return n
        return None

    pairs, only_a, matched_b = [], [], set()
    for a in a_list:
        b = lookup(opt.sheet_map.get(a, a))
        if b is None:
            only_a.append(a)
        else:
            pairs.append((a, b))
            matched_b.add(b)

    only_b = [n for n in names_b if n not in matched_b and keep(n)]
    return pairs, only_a, only_b


# ---------------------------------------------------------------------------
#  Cell comparison
# ---------------------------------------------------------------------------

def compare_sheet(sheet_label, grid_a, grid_b, opt: Options, budget: int):
    """Compare two cell grids positionally. Returns (diffs, truncated_flag, cells_compared)."""
    diffs: List[Dict[str, Any]] = []
    coords = set(grid_a) | set(grid_b)
    truncated = False
    limit = min(opt.max_diffs_per_sheet, budget)

    for (r, c) in sorted(coords):
        va, vb = grid_a.get((r, c)), grid_b.get((r, c))
        if values_equal(va, vb, opt):
            continue
        if len(diffs) >= limit:
            truncated = True
            break

        delta = pct = None
        if isinstance(va, (int, float)) and isinstance(vb, (int, float)) \
                and not isinstance(va, bool) and not isinstance(vb, bool):
            delta = float(vb) - float(va)
            if va != 0:
                pct = delta / abs(float(va))

        diffs.append({
            "Sheet": sheet_label,
            "Cell": f"{get_column_letter(c)}{r}",
            "Row": r,
            "Column": get_column_letter(c),
            "Column #": c,
            "Difference": difference_kind(va, vb, opt),
            "Value in A": va,
            "Value in B": vb,
            "Type in A": value_kind(va),
            "Type in B": value_kind(vb),
            "Delta (B-A)": delta,
            "% Change": pct,
        })

    return diffs, truncated, len(coords)


# ---------------------------------------------------------------------------
#  Orchestration
# ---------------------------------------------------------------------------

def compare_workbooks(file_a: str | os.PathLike, file_b: str | os.PathLike, opt: Options) -> Dict[str, Any]:
    started = dt.datetime.now()
    src_a = open_source(file_a, opt)
    warnings: List[str] = []

    try:
        src_b = open_source(file_b, opt)
    except Exception:
        src_a.close()
        raise

    try:
        # Dates: if one reader types them and the other hands back raw serial numbers,
        # compare both sides as serial numbers so the two are actually comparable.
        if opt.date_handling == "auto":
            mixed = src_a.typed_dates != src_b.typed_dates
            opt = replace(opt, date_handling="serial" if mixed else "native")
            if mixed:
                blind = src_a if not src_a.typed_dates else src_b
                warnings.append(
                    f"{blind.path.name} is read by {blind.engine}, which reports dates as raw Excel serial "
                    "numbers rather than dates. Both workbooks are therefore compared as serial numbers "
                    "(date_handling='serial'), so dates still compare correctly - but the report shows that "
                    "side's dates as numbers. Convert it to .xlsx to see real dates."
                )
        for s in (src_a, src_b):
            s.opt = opt
            if not s.typed_dates and opt.date_handling == "native":
                warnings.append(f"{s.path.name}: {s.engine} reports dates as Excel serial numbers.")
            if opt.check_cached_values and not s.has_formulas:
                warnings.append(f"{s.path.name}: {s.engine} cannot see formulas, so the "
                                "uncalculated-formula check was skipped for this workbook.")
            if s.note and s.origin is None and s.engine != "openpyxl":
                warnings.append(f"{s.path.name}: {s.note}")

        pairs, only_a, only_b = pair_sheets(src_a.sheets, src_b.sheets, opt)
        all_diffs: List[Dict[str, Any]] = []
        inventory: List[Dict[str, Any]] = []
        truncated_any = False

        for name_a, name_b in pairs:
            ga, ra, ca = src_a.grid(name_a)
            gb, rb, cb = src_b.grid(name_b)

            label = name_a if name_a == name_b else f"{name_a} -> {name_b}"
            budget = max(0, opt.max_total_diffs - len(all_diffs))
            diffs, truncated, compared = compare_sheet(label, ga, gb, opt, budget)
            truncated_any = truncated_any or truncated
            all_diffs.extend(diffs)

            if opt.check_cached_values:
                for src, nm, grid, side in ((src_a, name_a, ga, "A"), (src_b, name_b, gb, "B")):
                    total, missing, samples = src.formula_scan(nm, grid)
                    if missing:
                        warnings.append(
                            f"Workbook {side}, sheet '{nm}': {missing:,} of {total:,} formula cells have no "
                            f"cached value (e.g. {', '.join(samples)}). They compare as blank. "
                            "Open the file in Excel and re-save it to store calculated values."
                        )

            inventory.append({
                "Sheet in A": name_a,
                "Sheet in B": name_b,
                "Status": "Compared" + (" (truncated)" if truncated else ""),
                "Used range A": f"{ra} rows x {ca} cols" if ra else "empty",
                "Used range B": f"{rb} rows x {cb} cols" if rb else "empty",
                "Populated cells A": len(ga),
                "Populated cells B": len(gb),
                "Cells compared": compared,
                "Differences": len(diffs),
            })
            del ga, gb

        for name in only_a:
            g, r, c = src_a.grid(name)
            inventory.append({
                "Sheet in A": name, "Sheet in B": "", "Status": "Only in A",
                "Used range A": f"{r} rows x {c} cols" if r else "empty", "Used range B": "",
                "Populated cells A": len(g), "Populated cells B": 0,
                "Cells compared": 0, "Differences": "",
            })
        for name in only_b:
            g, r, c = src_b.grid(name)
            inventory.append({
                "Sheet in A": "", "Sheet in B": name, "Status": "Only in B",
                "Used range A": "", "Used range B": f"{r} rows x {c} cols" if r else "empty",
                "Populated cells A": 0, "Populated cells B": len(g),
                "Cells compared": 0, "Differences": "",
            })

        def note_only(names, side, status):
            for name in names:
                inventory.append({
                    "Sheet in A": name if side == "A" else "", "Sheet in B": name if side == "B" else "",
                    "Status": status, "Used range A": "", "Used range B": "",
                    "Populated cells A": 0, "Populated cells B": 0,
                    "Cells compared": 0, "Differences": "",
                })

        note_only(src_a.chart_sheets, "A", "Chart sheet (not compared)")
        note_only(src_b.chart_sheets, "B", "Chart sheet (not compared)")
        note_only(src_a.hidden_sheets, "A", "Hidden (skipped)")
        note_only(src_b.hidden_sheets, "B", "Hidden (skipped)")

        if truncated_any:
            warnings.append("Difference limit reached - the report is incomplete. "
                            "Raise max_diffs_per_sheet / max_total_diffs, or narrow the sheet list.")

        return {
            "file_a": str(Path(file_a).resolve()),
            "file_b": str(Path(file_b).resolve()),
            "source_a": f"{src_a.engine}" + (f" (after {src_a.note})" if src_a.origin else ""),
            "source_b": f"{src_b.engine}" + (f" (after {src_b.note})" if src_b.origin else ""),
            "started": started,
            "elapsed": (dt.datetime.now() - started).total_seconds(),
            "pairs": pairs,
            "only_in_a": only_a,
            "only_in_b": only_b,
            "chart_sheets": {"A": src_a.chart_sheets, "B": src_b.chart_sheets},
            "inventory": inventory,
            "differences": all_diffs,
            "warnings": warnings,
            "truncated": truncated_any,
            "options": opt,
            "identical": not all_diffs and not only_a and not only_b,
        }
    finally:
        src_a.close()
        src_b.close()

## 5. Report writer
Turns the result into the output workbook: `Summary`, `Sheet Inventory`, `Cell Differences`,
and optional per-sheet tabs. Differences are colour-coded — yellow *changed*, orange *type changed*,
green *added in B*, red *removed in B*.

In [ ]:
HEADER_FILL = PatternFill("solid", fgColor="1F3864")
HEADER_FONT = Font(color="FFFFFF", bold=True)
TITLE_FONT = Font(bold=True, size=13, color="1F3864")
LABEL_FONT = Font(bold=True)
THIN = Side(style="thin", color="D9D9D9")
CELL_BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

ROW_FILLS = {
    "Value changed": PatternFill("solid", fgColor="FFF2CC"),
    "Type changed":  PatternFill("solid", fgColor="FCE4D6"),
    "Added in B":    PatternFill("solid", fgColor="E2EFDA"),
    "Removed in B":  PatternFill("solid", fgColor="FBE5E5"),
}

DIFF_COLUMNS = ["Sheet", "Cell", "Row", "Column", "Column #", "Difference",
                "Value in A", "Value in B", "Type in A", "Type in B",
                "Delta (B-A)", "% Change"]

WRITABLE = (str, int, float, dt.datetime, dt.date, dt.time, dt.timedelta, bool)


def safe_value(v: Any, opt: Options) -> Any:
    """Coerce any cell value into something openpyxl will accept, without it becoming a formula."""
    if v is None or isinstance(v, bool) or isinstance(v, (int, float)):
        if isinstance(v, float) and (math.isnan(v) or math.isinf(v)):
            return str(v)
        return v
    if isinstance(v, dt.datetime):
        return v.replace(tzinfo=None) if v.tzinfo else v
    if isinstance(v, (dt.date, dt.time, dt.timedelta)):
        return v
    s = v if isinstance(v, str) else repr(v)
    s = ILLEGAL_CHARACTERS_RE.sub("", s)
    if len(s) > opt.max_value_chars:
        s = s[: opt.max_value_chars] + f"... [{len(s):,} chars]"
    return s


def put(ws, row: int, col: int, value: Any, opt: Options):
    """Write a value, forcing text so a leading '=' or '+' never turns into a formula."""
    v = safe_value(value, opt)
    cell = ws.cell(row=row, column=col, value=v)
    if isinstance(v, str) and v[:1] in {"=", "+", "-", "@"}:
        cell.data_type = "s"
    return cell


def autosize(ws, headers: Sequence[str], rows: Sequence[Sequence[Any]], cap: int = 55):
    for i, header in enumerate(headers, start=1):
        width = len(str(header))
        for r in rows[:400]:                      # sample the first 400 rows: enough, and fast
            if i - 1 < len(r) and r[i - 1] is not None:
                width = max(width, len(str(r[i - 1])))
        ws.column_dimensions[get_column_letter(i)].width = min(max(width + 2, 9), cap)


def write_table(ws, headers: Sequence[str], rows: Sequence[Sequence[Any]], opt: Options,
                start_row: int = 1, colour_by: Optional[int] = None):
    for i, header in enumerate(headers, start=1):
        c = ws.cell(row=start_row, column=i, value=header)
        c.fill, c.font = HEADER_FILL, HEADER_FONT
        c.alignment = Alignment(vertical="center")
        c.border = CELL_BORDER

    for j, data_row in enumerate(rows, start=start_row + 1):
        fill = ROW_FILLS.get(str(data_row[colour_by])) if colour_by is not None else None
        for i, value in enumerate(data_row, start=1):
            c = put(ws, j, i, value, opt)
            c.border = CELL_BORDER
            if fill is not None:
                c.fill = fill

    last_row = start_row + len(rows)
    if opt.freeze_and_filter and rows:
        ws.freeze_panes = ws.cell(row=start_row + 1, column=1)
        ws.auto_filter.ref = f"A{start_row}:{get_column_letter(len(headers))}{last_row}"
    autosize(ws, headers, rows)
    return last_row


def safe_title(name: str, used: set) -> str:
    clean = re.sub(r"[\\/*?:\[\]]", "_", str(name))[:28] or "Sheet"
    title, n = clean, 1
    while title.casefold() in used:
        suffix = f"_{n}"
        title = clean[: 31 - len(suffix)] + suffix
        n += 1
    used.add(title.casefold())
    return title


def build_report(result: Dict[str, Any], out_path: str | os.PathLike, opt: Options) -> Path:
    diffs = result["differences"]
    wb = Workbook()

    # ---- Summary -----------------------------------------------------------
    ws = wb.active
    ws.title = "Summary"
    ws.sheet_view.showGridLines = False
    ws["A1"] = "Excel workbook comparison - values only"
    ws["A1"].font = Font(bold=True, size=15, color="1F3864")

    by_kind: Dict[str, int] = {}
    for d in diffs:
        by_kind[d["Difference"]] = by_kind.get(d["Difference"], 0) + 1
    sheets_with_diffs = sorted({d["Sheet"] for d in diffs})

    verdict = "IDENTICAL - no value differences found" if result["identical"] else \
              f"DIFFERENT - {len(diffs):,} differing cells across {len(sheets_with_diffs)} sheet(s)"

    lines: List[Tuple[str, Any]] = [
        ("Result", verdict),
        ("", ""),
        ("Workbook A (before)", result["file_a"]),
        ("  read with", result["source_a"]),
        ("Workbook B (after)", result["file_b"]),
        ("  read with", result["source_b"]),
        ("Report generated", result["started"].strftime("%Y-%m-%d %H:%M:%S")),
        ("Comparison time", f"{result['elapsed']:.2f} s"),
        ("", ""),
        ("Sheets compared", len(result["pairs"])),
        ("Sheets only in A", ", ".join(result["only_in_a"]) or "none"),
        ("Sheets only in B", ", ".join(result["only_in_b"]) or "none"),
        ("Chart sheets skipped", ", ".join(result["chart_sheets"]["A"] + result["chart_sheets"]["B"]) or "none"),
        ("Sheets with differences", ", ".join(sheets_with_diffs) or "none"),
        ("", ""),
        ("Total differences", len(diffs)),
    ]
    for kind in ("Value changed", "Type changed", "Added in B", "Removed in B"):
        lines.append((f"  {kind}", by_kind.get(kind, 0)))

    lines += [("", ""), ("Comparison rules applied", "")]
    for k, v in asdict(opt).items():
        lines.append((f"  {k}", "none" if v is None else (str(v) if not isinstance(v, (int, float, bool)) else v)))

    if result["warnings"]:
        lines += [("", ""), ("Warnings", "")]
        lines += [(f"  {i}", w) for i, w in enumerate(result["warnings"], start=1)]

    lines += [
        ("", ""),
        ("Not compared", "formulas (cached values are used instead), charts, images, pivot caches, "
                         "number formats, fonts, fills, column widths, comments, macros"),
        ("Matching", "cell-for-cell by grid position - an inserted row shifts every cell below it"),
    ]

    row = 3
    for label, value in lines:
        c = ws.cell(row=row, column=1, value=label)
        c.font = LABEL_FONT if label and not label.startswith(" ") else Font()
        put(ws, row, 2, value, opt).alignment = Alignment(wrap_text=False, vertical="top")
        row += 1
    ws.column_dimensions["A"].width = 30
    ws.column_dimensions["B"].width = 110
    ws["B3"].font = Font(bold=True, color="C00000" if not result["identical"] else "1E7145")

    # ---- Sheet Inventory ---------------------------------------------------
    inv = result["inventory"]
    ws_inv = wb.create_sheet("Sheet Inventory")
    inv_headers = ["Sheet in A", "Sheet in B", "Status", "Used range A", "Used range B",
                   "Populated cells A", "Populated cells B", "Cells compared", "Differences"]
    write_table(ws_inv, inv_headers, [[r.get(h, "") for h in inv_headers] for r in inv], opt)

    # ---- Cell Differences --------------------------------------------------
    ws_diff = wb.create_sheet("Cell Differences")
    rows = [[d[h] for h in DIFF_COLUMNS] for d in diffs]
    if rows:
        last = write_table(ws_diff, DIFF_COLUMNS, rows, opt,
                           colour_by=DIFF_COLUMNS.index("Difference"))
        pct_col = get_column_letter(DIFF_COLUMNS.index("% Change") + 1)
        for r in range(2, last + 1):
            ws_diff[f"{pct_col}{r}"].number_format = "0.00%"
    else:
        write_table(ws_diff, DIFF_COLUMNS, [], opt)
        ws_diff["A2"] = "No differing cells."

    # ---- Optional per-sheet tabs ------------------------------------------
    if opt.per_sheet_tabs and diffs:
        used = {"summary", "sheet inventory", "cell differences"}
        per_sheet_cols = [c for c in DIFF_COLUMNS if c != "Sheet"]
        for sheet_name in sheets_with_diffs:
            subset = [[d[c] for c in per_sheet_cols] for d in diffs if d["Sheet"] == sheet_name]
            tab = wb.create_sheet(safe_title(sheet_name, used))
            tab["A1"] = f"Differences in sheet: {sheet_name}"
            tab["A1"].font = TITLE_FONT
            write_table(tab, per_sheet_cols, subset, opt, start_row=3,
                        colour_by=per_sheet_cols.index("Difference"))

    out = Path(out_path)
    if out.parent and not out.parent.exists():
        out.parent.mkdir(parents=True, exist_ok=True)
    wb.save(out)
    return out.resolve()


def print_report(result: Dict[str, Any], out_path: Optional[Path] = None) -> None:
    diffs = result["differences"]
    bar = "=" * 78
    print(bar)
    print(f"A: {result['file_a']}")
    print(f"   read with {result['source_a']}")
    print(f"B: {result['file_b']}")
    print(f"   read with {result['source_b']}")
    print(bar)
    if result["identical"]:
        print("No value differences found.")
    else:
        print(f"{len(diffs):,} differing cells in {len({d['Sheet'] for d in diffs})} sheet(s)"
              f"  |  compared in {result['elapsed']:.2f}s")
        counts: Dict[str, int] = {}
        for d in diffs:
            counts[d["Sheet"]] = counts.get(d["Sheet"], 0) + 1
        for sheet, n in sorted(counts.items(), key=lambda kv: -kv[1]):
            print(f"   {n:>9,}  {sheet}")
    if result["only_in_a"]:
        print(f"Sheets only in A: {', '.join(result['only_in_a'])}")
    if result["only_in_b"]:
        print(f"Sheets only in B: {', '.join(result['only_in_b'])}")
    for w in result["warnings"]:
        print(f"WARNING: {w}")
    if out_path:
        print(bar)
        print(f"Report written to {out_path}")

## 6. Run the comparison

In [ ]:
if not (Path(FILE_A).exists() and Path(FILE_B).exists()):
    result = None
    missing = [p for p in (FILE_A, FILE_B) if not Path(p).exists()]
    print(f"Not found: {', '.join(missing)}")
    print("Point FILE_A / FILE_B at your workbooks in section 2, then re-run this cell.")
    print("Section 7 runs the whole pipeline on generated sample files if you just want to see it work.")
else:
    result = compare_workbooks(FILE_A, FILE_B, OPT)
    report_path = build_report(result, OUTPUT_FILE, OPT)
    print_report(result, report_path)

## 7. Explore the differences here
Optional. The same rows that went into the report, as a DataFrame you can slice.

In [ ]:
if result is None:
    print("Run section 5 first (it needs FILE_A and FILE_B to exist).")
elif pd is None:
    print("pandas is not installed - `pip install pandas` to use this section.")
else:
    diffs_df = pd.DataFrame(result["differences"], columns=DIFF_COLUMNS)
    print(f"{len(diffs_df):,} differences\n")

    if not diffs_df.empty:
        print("By sheet and type")
        print(diffs_df.pivot_table(index="Sheet", columns="Difference",
                                   values="Cell", aggfunc="count", fill_value=0))

        numeric = diffs_df[diffs_df["Delta (B-A)"].notna()]
        if not numeric.empty:
            print("\nLargest numeric movements")
            top = numeric.reindex(numeric["Delta (B-A)"].abs().sort_values(ascending=False).index)
            print(top[["Sheet", "Cell", "Value in A", "Value in B", "Delta (B-A)", "% Change"]]
                  .head(15).to_string(index=False))

    # Slice it however you like:
    #   diffs_df.query("Sheet == 'Sales' and Difference == 'Value changed'")
    #   diffs_df[diffs_df["Delta (B-A)"].abs() > 1000]
    diffs_df.head(25)

## 8. Self-test on generated workbooks
Builds two small workbooks with known differences, runs the full pipeline, and asserts the expected
differences (and only those) come back — plus a few unit checks on the date/serial handling. Use it to
confirm the notebook works before pointing it at real files, and as a worked example of what each
difference type looks like.

In [ ]:
RUN_SELF_TEST = True     # set to False once you are pointing at your own workbooks

if RUN_SELF_TEST:
    import tempfile

    demo_dir = Path(tempfile.mkdtemp(prefix="xlsx_diff_demo_"))

    def build_demo(path: Path, variant: str) -> None:
        wb = Workbook()

        sales = wb.active
        sales.title = "Sales"
        rows = [
            ["Region", "Units", "Price", "Revenue", "Updated"],
            ["North", 100, 9.99, 999.00, dt.datetime(2026, 1, 31)],
            ["South", 250, 9.99, 2497.50, dt.datetime(2026, 1, 31)],
            ["East",  180, 9.99, 1798.20, dt.datetime(2026, 1, 31)],
            ["West",  None, 9.99, None,   dt.datetime(2026, 1, 31)],
        ]
        if variant == "B":
            rows[2][1] = 260              # C: Units 250 -> 260   (value changed)
            rows[2][3] = 2597.40          #    Revenue follows    (value changed)
            rows[3][0] = "  East  "       # A: padding only       -> equal, trim_whitespace
            rows[4][1] = 75               # B: blank -> 75        (added in B)
            rows[1][2] = "9.99"           # C: number -> text     (type changed)
        for r in rows:
            sales.append(r)
        if variant == "A":
            sales["G2"] = "note to delete"    # removed in B

        formulas = wb.create_sheet("Formulas")
        formulas["A1"], formulas["A2"] = 10, 20
        formulas["A3"] = "=A1+A2"             # written by openpyxl -> no cached value, triggers the warning

        if variant == "A":
            legacy = wb.create_sheet("Legacy Notes")
            legacy["A1"] = "dropped in the new version"
        else:
            extra = wb.create_sheet("New Tab")
            extra["A1"] = "added in the new version"

        wb.save(path)

    demo_a, demo_b = demo_dir / "workbook_a.xlsx", demo_dir / "workbook_b.xlsx"
    build_demo(demo_a, "A")
    build_demo(demo_b, "B")

    demo_result = compare_workbooks(demo_a, demo_b, OPT)
    demo_out = build_report(demo_result, demo_dir / "differences.xlsx", OPT)
    print_report(demo_result, demo_out)

    found = {(d["Sheet"], d["Cell"], d["Difference"]) for d in demo_result["differences"]}
    expected = {
        ("Sales", "B3", "Value changed"),   # 250 -> 260
        ("Sales", "D3", "Value changed"),   # 2497.50 -> 2597.40
        ("Sales", "B5", "Added in B"),      # blank -> 75
        ("Sales", "C2", "Type changed"),    # 9.99 -> "9.99"
        ("Sales", "G2", "Removed in B"),    # note deleted
    }
    print("\nself-test")
    assert expected <= found, f"missing: {expected - found}"
    assert not [d for d in demo_result["differences"] if d["Cell"] == "A4"], \
        "whitespace-only change should not be reported while trim_whitespace=True"
    assert demo_result["only_in_a"] == ["Legacy Notes"], demo_result["only_in_a"]
    assert demo_result["only_in_b"] == ["New Tab"], demo_result["only_in_b"]
    assert any("cached value" in w for w in demo_result["warnings"]), "expected the cached-value warning"

    # Date <-> Excel serial handling, which is what makes a mixed .xlsb/.xlsx run correct.
    # 36528.0 is the serial a real Excel-written file uses for 2000-01-03.
    assert to_excel_serial(dt.datetime(1900, 3, 1)) == 61.0
    assert to_excel_serial(dt.datetime(2000, 1, 3)) == 36528.0
    assert to_excel_serial(dt.datetime(2026, 9, 7, 12, 0)) == 46272.5
    assert to_excel_serial(dt.date(2025, 7, 4)) == 45842.0

    serial = replace(OPT, date_handling="serial")
    native = replace(OPT, date_handling="native")
    assert values_equal(dt.datetime(2000, 1, 3), 36528.0, serial), "serial mode should bridge the formats"
    assert not values_equal(dt.datetime(2000, 1, 4), 36528.0, serial)
    assert not values_equal(dt.datetime(2000, 1, 3), 36528.0, native), "native mode keeps types apart"
    assert values_equal(dt.datetime(2000, 1, 3, 9, 30), 36528.0, replace(serial, ignore_time_component=True))
    assert difference_kind(dt.datetime(2000, 1, 4), 36528.0, serial) == "Value changed"
    assert difference_kind(dt.datetime(2000, 1, 4), 36528.0, native) == "Type changed"
    assert XLSB_ERRORS["0x2a"] == "#N/A"

    print("  all checks passed - engine, report writer, dates and warnings behave as documented")
    print(f"  demo files: {demo_dir}")